<a href="https://colab.research.google.com/github/ldongheedev/-BDA-LLM-RAG-Program/blob/main/6%EC%A3%BC%EC%B0%A8_%EB%B3%B5%EC%8A%B5%EA%B3%BC%EC%A0%9C_%EB%94%A5%EB%9F%AC%EB%8B%9D%EC%BD%94%EB%93%9C%EC%9D%B4%ED%95%B4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6주차 복습 과제: 딥러닝 코드에 대한 이해 및 습득

> **과제 목표**
> 1. ChatGPT가 생겨날 수 있는 이유는 딥러닝의 발전임을 인식
> 2. 딥러닝의 기본 구조를 파악하고, 코드로 이해함을 습득
> 3. 이러한 구조가 더 고도화되면 단어를 빠르게 추천하고, 그를 잇는 문장이 바로 챗봇의 응답화가 됨을 이해


---
## 1. ChatGPT는 왜 생겨날 수 있었는가? — 딥러닝의 발전

### 1-1. 전통 ML → 딥러닝 → ChatGPT로 이어지는 흐름

```
[전통 ML]          [딥러닝]              [ChatGPT]
TF-IDF + 분류기 →  RNN/LSTM →  Transformer → GPT
  (3~4주차)         (6주차)      (7주차)       (8주차)

단어 빈도 세기  → 순서/문맥 이해  →  모든 단어를  →  대규모 학습으로
                                   동시에 파악      자연스러운 문장 생성
```

ChatGPT의 핵심은 **"다음 단어 예측"**입니다.
이것이 가능해진 이유는 딥러닝이 **단어의 순서와 문맥**을 이해할 수 있게 되었기 때문입니다.

| 발전 단계 | 핵심 기술 | 할 수 있는 것 |
|-----------|-----------|---------------|
| TF-IDF + ML | 단어 빈도 수치화 | 단순 분류 (스팸/정상) |
| Word2Vec | 단어 임베딩 | 단어 간 의미 유사도 파악 |
| RNN/LSTM | 순서 반영 신경망 | 문맥을 고려한 분류/생성 |
| Transformer | Self-Attention | 긴 문장도 빠르고 정확하게 이해 |
| GPT | 대규모 사전학습 | 자연스러운 대화 (ChatGPT) |


### 1-2. 전통 ML의 한계 — 왜 딥러닝이 필요한가

아래 코드로 TF-IDF/BoW의 근본적 한계를 확인합니다.


In [ ]:
# 한계 1: 단어 순서를 무시한다
from collections import Counter

sentence_a = "영화가 지루하지 않고 재미있다"   # 긍정
sentence_b = "영화가 재미있지 않고 지루하다"   # 부정

bow_a = Counter(sentence_a.split())
bow_b = Counter(sentence_b.split())

print(f"문장A (긍정): {bow_a}")
print(f"문장B (부정): {bow_b}")
print()
print("결론: 거의 같은 단어 구성 → BoW/TF-IDF로는 긍정/부정 구별 불가!")
print("→ '않고'의 위치에 따라 의미가 완전히 반대인데, 순서를 무시하기 때문")


In [ ]:
# 한계 2: 문맥을 이해하지 못한다
examples = [
    ("사과를 먹었다", "사과 = 과일"),
    ("진심으로 사과한다", "사과 = 사죄"),
]

print("같은 '사과'인데 의미가 다릅니다:")
for sentence, meaning in examples:
    print(f"  {sentence} → {meaning}")

print()
print("결론: TF-IDF는 '사과'를 항상 같은 단어로 취급!")
print("→ 주변 단어(문맥)를 보고 의미를 파악해야 하며, 이것이 딥러닝이 필요한 이유")


**정리: 전통 ML vs 딥러닝**

| 구분 | 전통 ML (TF-IDF + 분류기) | 딥러닝 (RNN/Transformer) |
|------|--------------------------|-------------------------|
| 단어 표현 | 빈도 기반 (희소 벡터) | 임베딩 (밀집 벡터) |
| 단어 순서 | ❌ 무시 | ✅ 반영 |
| 문맥 이해 | ❌ 불가 | ✅ 가능 |
| 특성 추출 | 사람이 직접 설계 | 모델이 자동 학습 |


---
## 2. 딥러닝의 기본 구조 — 코드로 이해하기

### 2-1. 신경망의 핵심: 입력 → 가중합 → 활성화 → 출력

```
    입력층          은닉층          출력층
   (Input)        (Hidden)       (Output)

   x1 ─────┐    ┌─── h1 ───┐
            ├────┤          ├──── y (긍정/부정)
   x2 ─────┤    └─── h2 ───┘
            │
   x3 ─────┘
         W1 (가중치)    W2 (가중치)
```

| 개념 | 설명 | 비유 |
|------|------|------|
| **가중치 (Weight)** | 입력에 곱해지는 학습 가능한 값 | 각 단어의 "중요도 점수" |
| **편향 (Bias)** | 가중합에 더해지는 상수 | 기본 점수 (가산점) |
| **활성화 함수** | 비선형 변환 (ReLU, Sigmoid 등) | "활성화시킬까?" 판단 |
| **손실 함수** | 예측과 정답의 차이 | 시험 점수의 감점 |


In [ ]:
# 뉴런의 기본 연산을 NumPy로 직접 구현
import numpy as np

# === 하나의 뉴런: y = activation(W · x + b) ===
x = np.array([0.5, -0.3, 0.8])  # 입력 벡터 (단어 임베딩 3차원이라 가정)
W = np.array([0.2, 0.7, -0.5])  # 가중치
b = 0.1                          # 편향

# Step 1: 가중합 (weighted sum)
z = np.dot(W, x) + b
print(f"입력 x: {x}")
print(f"가중치 W: {W}")
print(f"편향 b: {b}")
print(f"가중합 z = W·x + b = {z:.4f}")
print()
print("→ 가중합은 '입력값들을 중요도에 따라 합산'하는 과정")


### 2-2. 활성화 함수 — 비선형성을 부여하는 핵심


In [ ]:
# 활성화 함수: Sigmoid, ReLU, Tanh 비교
import numpy as np
import matplotlib.pyplot as plt

z = np.linspace(-5, 5, 100)

# 세 가지 활성화 함수 정의
sigmoid = 1 / (1 + np.exp(-z))    # 출력 범위: 0 ~ 1
relu = np.maximum(0, z)            # 출력 범위: 0 ~ 무한대
tanh = np.tanh(z)                  # 출력 범위: -1 ~ 1

plt.figure(figsize=(10, 4))
plt.plot(z, sigmoid, label="Sigmoid (0~1) — 출력층에서 확률 계산용", linewidth=2)
plt.plot(z, relu, label="ReLU (0~inf) — 은닉층에서 가장 많이 사용", linewidth=2)
plt.plot(z, tanh, label="Tanh (-1~1) — RNN 계열에서 사용", linewidth=2)
plt.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
plt.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
plt.legend(fontsize=10)
plt.title("활성화 함수 비교", fontsize=14)
plt.xlabel("z (가중합)")
plt.ylabel("출력값")
plt.grid(True, alpha=0.3)
plt.show()

print("Sigmoid: 마지막 출력층에서 확률(긍정/부정)을 구할 때 사용")
print("ReLU: 은닉층에서 주로 사용 (계산이 빠르고 학습이 잘 됨)")
print("Tanh: RNN 내부에서 은닉 상태를 계산할 때 사용")


### 2-3. 손실 함수 — 예측이 얼마나 틀렸는지 측정


In [ ]:
# 손실 함수: Binary Cross-Entropy
import numpy as np

def binary_cross_entropy(y_true, y_pred):
    y_pred = np.clip(y_pred, 1e-7, 1 - 1e-7)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# 정답이 긍정(1)일 때, 다양한 예측값에 대한 손실
y_true = 1
print("정답이 '긍정(1)'일 때:")
print(f"  예측 0.9 (잘 맞춤) → 손실: {binary_cross_entropy(y_true, 0.9):.4f}")
print(f"  예측 0.5 (애매함)  → 손실: {binary_cross_entropy(y_true, 0.5):.4f}")
print(f"  예측 0.1 (틀림)   → 손실: {binary_cross_entropy(y_true, 0.1):.4f}")
print()
print("→ 예측이 정답에 가까울수록 손실이 작다!")
print("→ 학습 = 이 손실을 줄이는 방향으로 가중치를 업데이트하는 것")


### 2-4. 학습 과정 = 경사 하강법

```
1. 입력 데이터를 신경망에 넣는다           (순전파, Forward)
2. 예측값과 정답을 비교하여 손실을 계산한다  (Loss 계산)
3. 손실을 줄이는 방향으로 가중치를 조정한다  (역전파, Backward)
4. 1~3을 반복한다                         (학습 루프)
```

비유: 눈을 감고 산에서 내려오기
→ 발밑의 경사(기울기)를 느끼며, 가장 가파르게 내려가는 방향으로 한 발씩 이동
→ 가장 낮은 곳(최소 손실)에 도달하는 것이 목표


In [ ]:
# PyTorch 자동 미분 — 학습의 핵심 원리
import torch

# requires_grad=True → 이 텐서에 대한 기울기를 추적
w = torch.tensor(2.0, requires_grad=True)
x = torch.tensor(3.0)

# 순전파: y = w * x
y = w * x
print(f"w = {w.item()}, x = {x.item()}")
print(f"y = w * x = {y.item()}")

# 역전파: dy/dw 계산
y.backward()
print(f"dy/dw = {w.grad.item()}")  # x의 값인 3.0

print()
print("w.grad = 3.0 → w를 1 늘리면 y가 3 늘어난다는 뜻")
print("이 기울기 정보로 가중치를 업데이트하는 것이 '학습'!")


### 2-5. PyTorch로 간단한 신경망 만들기


In [ ]:
# nn.Module로 2층 신경망 구현
import torch
import torch.nn as nn

class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.layer1 = nn.Linear(input_size, hidden_size)   # 입력→은닉
        self.relu = nn.ReLU()                               # 활성화
        self.layer2 = nn.Linear(hidden_size, output_size)  # 은닉→출력
        self.sigmoid = nn.Sigmoid()                         # 확률 출력

    def forward(self, x):
        x = self.layer1(x)    # 가중합
        x = self.relu(x)      # 활성화 (비선형 변환)
        x = self.layer2(x)    # 가중합
        x = self.sigmoid(x)   # 0~1 확률로 변환
        return x

# 모델 생성: 입력 10차원, 은닉 5차원, 출력 1차원
model = SimpleNN(input_size=10, hidden_size=5, output_size=1)
print(model)

# 테스트 입력
dummy_input = torch.randn(1, 10)
output = model(dummy_input)
print(f"\n입력 형태: {dummy_input.shape}")
print(f"출력 (확률): {output.item():.4f}")
print()
print("→ 아직 학습 전이라 출력은 무작위. 학습을 통해 정확해짐")


---
## 3. RNN과 LSTM — 순서를 이해하는 신경망

### 3-1. RNN의 핵심 아이디어

일반 신경망은 입력을 독립적으로 처리하지만, 텍스트는 **순서**가 중요합니다.
RNN은 이전 단어의 정보를 **기억**하면서 다음 단어를 처리합니다.

```
"나는   영화를   정말    좋아한다"
  ↓      ↓       ↓        ↓
[RNN] → [RNN] → [RNN] → [RNN] → 긍정!
  h0      h1      h2      h3

h (hidden state) = "지금까지 읽은 내용의 요약"
→ 사람이 문장을 왼쪽에서 오른쪽으로 읽으며 의미를 누적하는 것과 동일
```

### 3-2. RNN의 한계 → LSTM으로 해결

문장이 길어지면 앞쪽 정보가 점점 사라지는 **장기 의존성 문제**(기울기 소실)가 발생합니다.
LSTM은 3개의 **게이트(Gate)**로 이 문제를 해결합니다:

| 게이트 | 역할 | 비유 |
|--------|------|------|
| 망각 게이트 | 이전 기억 중 불필요한 것을 삭제 | 노트에서 줄 긋기 |
| 입력 게이트 | 새로운 중요 정보를 기억에 추가 | 중요한 내용 적기 |
| 출력 게이트 | 현재 필요한 정보를 출력 | 답안지에 쓸 것 선택 |


In [ ]:
# RNN 동작을 NumPy로 직접 구현하여 이해
import numpy as np

np.random.seed(42)

embedding_dim = 4   # 단어 임베딩 차원
hidden_dim = 3      # 은닉 상태 차원

# 가중치 (실제로는 학습되지만, 여기선 랜덤 초기화)
W_xh = np.random.randn(hidden_dim, embedding_dim) * 0.1
W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.1
b = np.zeros(hidden_dim)

# 가상의 문장: "나는 영화를 좋아한다" (3단어, 각 4차원 임베딩)
sentence = {
    "나는":     np.array([0.5, -0.3, 0.8, 0.1]),
    "영화를":   np.array([0.2, 0.7, -0.1, 0.4]),
    "좋아한다": np.array([0.9, 0.3, 0.5, -0.2]),
}

# RNN 순전파
h = np.zeros(hidden_dim)  # 초기 은닉 상태

print("=== RNN 순전파 과정 ===")
print(f"초기 h_0: {h}\n")

for t, (word, x_t) in enumerate(sentence.items()):
    # h_t = tanh(W_xh · x_t + W_hh · h_{t-1} + b)
    h = np.tanh(W_xh @ x_t + W_hh @ h + b)
    print(f"시점 {t+1}: '{word}'")
    print(f"  입력: {x_t}")
    print(f"  은닉 상태: {np.round(h, 4)}")
    print()

print(f"최종 은닉 상태 h_3: {np.round(h, 4)}")
print("→ 이 벡터가 전체 문장의 '요약'")
print("→ 이걸 분류층(Linear)에 넣으면 긍정/부정 판단 가능")


---
## 4. 실전: LSTM 감성 분류 모델 구현

지금까지 배운 모든 것을 합쳐서 **영화 리뷰 → 긍정/부정 분류** 모델을 만듭니다.

```
텍스트 → 토큰화 → 정수 인코딩 → 패딩 → 임베딩 → LSTM → 분류
                                                          ↓
"재미있다"                                            긍정 (0.92)
```

### 4-1. 데이터 준비


In [ ]:
# 학습 데이터 준비
import torch
import torch.nn as nn

reviews = [
    ("이 영화 정말 재미있다 최고의 영화", 1),
    ("배우 연기가 훌륭하고 스토리도 좋다", 1),
    ("감동적인 영화 눈물이 났다", 1),
    ("올해 최고의 영화 강추한다", 1),
    ("재미있고 감동적이다 또 보고 싶다", 1),
    ("연출이 뛰어나고 음악도 좋다", 1),
    ("완벽한 영화 모든 것이 좋았다", 1),
    ("시간 가는 줄 모르고 봤다 재미있다", 1),
    ("이 영화 정말 지루하다 별로다", 0),
    ("스토리가 너무 진부하고 재미없다", 0),
    ("시간 낭비 돈 아깝다", 0),
    ("연기가 어색하고 내용이 별로다", 0),
    ("최악의 영화 다시는 안 본다", 0),
    ("지루하고 졸렸다 비추천한다", 0),
    ("기대했는데 실망이다 별로다", 0),
    ("돈이 아깝다 최악이다", 0),
]

print(f"전체 데이터: {len(reviews)}개 (긍정 {sum(1 for _,l in reviews if l==1)}개 / 부정 {sum(1 for _,l in reviews if l==0)}개)")


### 4-2. 텍스트 → 숫자 변환 (어휘 사전 + 정수 인코딩 + 패딩)


In [ ]:
# 어휘 사전 구축
all_words = []
for text, _ in reviews:
    all_words.extend(text.split())

vocab = sorted(set(all_words))
word2idx = {word: idx + 2 for idx, word in enumerate(vocab)}
word2idx["<PAD>"] = 0  # 패딩 토큰 (빈칸)
word2idx["<UNK>"] = 1  # 미등록 단어

print(f"어휘 크기: {len(word2idx)}개")
print(f"어휘 사전 (처음 10개):")
for word, idx in list(word2idx.items())[:10]:
    print(f"  '{word}' → {idx}")


In [ ]:
# 문장 → 정수 시퀀스 변환 + 패딩
def encode_sentence(text, word2idx, max_len=10):
    tokens = text.split()
    indices = [word2idx.get(w, word2idx["<UNK>"]) for w in tokens]
    if len(indices) < max_len:
        indices = indices + [0] * (max_len - len(indices))  # 짧으면 0으로 채움
    else:
        indices = indices[:max_len]  # 길면 자름
    return indices

# 전체 데이터를 텐서로 변환
MAX_LEN = 10
X_data = [encode_sentence(text, word2idx, MAX_LEN) for text, _ in reviews]
y_data = [label for _, label in reviews]

X_tensor = torch.tensor(X_data, dtype=torch.long)
y_tensor = torch.tensor(y_data, dtype=torch.float32)

print(f"X 형태: {X_tensor.shape}")  # (16, 10) — 16문장, 최대 10단어
print(f"y 형태: {y_tensor.shape}")
print(f"\n첫 번째 문장 인코딩: {X_tensor[0]}")
print(f"첫 번째 레이블: {y_tensor[0]} (긍정)")


### 4-3. LSTM 감성 분류 모델 정의


In [ ]:
# LSTM 감성 분류 모델
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        # 1. 임베딩: 정수 → 밀집 벡터
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=0
        )
        # 2. LSTM: 순서를 반영하여 문장 이해
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        # 3. 분류층: LSTM 출력 → 긍정/부정 확률
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)              # (배치, 문장길이, embedding_dim)
        lstm_out, (h_n, c_n) = self.lstm(embedded) # h_n: 마지막 은닉 상태
        last_hidden = h_n.squeeze(0)               # (배치, hidden_dim)
        output = self.fc(last_hidden)              # (배치, 1)
        output = self.sigmoid(output)              # 0~1 확률
        return output.squeeze(1)

# 모델 생성
VOCAB_SIZE = len(word2idx)
EMBEDDING_DIM = 16
HIDDEN_DIM = 32

model = SentimentLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM)
print(model)
print(f"\n총 파라미터 수: {sum(p.numel() for p in model.parameters()):,}개")

print()
print("모델 구조 해설:")
print("  [정수 시퀀스] → [Embedding 16차원] → [LSTM 32차원] → [Linear+Sigmoid] → 확률")


### 4-4. 모델 학습 + 학습 과정 시각화


In [ ]:
# 학습 루프 + 손실/정확도 기록
import matplotlib.pyplot as plt

criterion = nn.BCELoss()  # Binary Cross-Entropy 손실 함수
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

EPOCHS = 100
loss_history = []
acc_history = []

print("=== 학습 시작 ===")
for epoch in range(EPOCHS):
    model.train()

    # 순전파
    predictions = model(X_tensor)              # 1. 모델 예측
    loss = criterion(predictions, y_tensor)    # 2. 손실 계산

    # 역전파
    optimizer.zero_grad()   # 3. 기울기 초기화
    loss.backward()         # 4. 역전파 (기울기 계산)
    optimizer.step()        # 5. 가중치 업데이트

    # 정확도 계산
    predicted_labels = (predictions >= 0.5).float()
    accuracy = (predicted_labels == y_tensor).float().mean()

    loss_history.append(loss.item())
    acc_history.append(accuracy.item())

    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/{EPOCHS} | 손실: {loss.item():.4f} | 정확도: {accuracy.item():.2%}")

print("\n=== 학습 완료! ===")


In [ ]:
# 학습 과정 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(loss_history, color="red", linewidth=1.5)
ax1.set_title("학습 손실 (Loss) — 낮을수록 좋음", fontsize=12)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3)

ax2.plot(acc_history, color="blue", linewidth=1.5)
ax2.set_title("학습 정확도 (Accuracy) — 높을수록 좋음", fontsize=12)
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_ylim([0, 1.05])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"최종 손실: {loss_history[-1]:.4f}")
print(f"최종 정확도: {acc_history[-1]:.2%}")


### 4-5. 새로운 문장으로 모델 테스트


In [ ]:
# 감성 예측 함수
def predict_sentiment(text, model, word2idx, max_len=10):
    model.eval()
    encoded = encode_sentence(text, word2idx, max_len)
    input_tensor = torch.tensor([encoded], dtype=torch.long)
    with torch.no_grad():
        prob = model(input_tensor).item()
    sentiment = "긍정" if prob >= 0.5 else "부정"
    return sentiment, prob

# 테스트
test_sentences = [
    "정말 재미있다 최고의 영화",
    "지루하고 별로다 실망이다",
    "연기가 좋고 감동적이다",
    "시간 낭비 최악이다",
    "영화 재미있고 좋다",
]

print("=== 감성 분석 결과 ===")
for sent in test_sentences:
    sentiment, prob = predict_sentiment(sent, model, word2idx)
    emoji = '😊' if sentiment == '긍정' else '😞'
    print(f"  '{sent}' → {sentiment} {emoji} (확률: {prob:.2%})")


---
## 5. 딥러닝 구조의 고도화 → 챗봇 응답이 되는 원리

### 5-1. LSTM에서 GPT로 이어지는 핵심 원리: "다음 단어 예측"

```
[LSTM 감성 분류]                    [GPT 챗봇]
"영화가 재미있다" → 긍정(0.92)     "오늘 날씨" → "가" → "좋다" → "."
                                     ↑
입력 전체를 보고     →    입력을 보고 다음 단어를 하나씩 생성
하나의 분류 결과              반복하면 문장이 완성
```

우리가 만든 LSTM 모델은 문장 전체를 읽고 **하나의 값**(긍정/부정)을 출력합니다.
이 구조를 바꿔서 **다음 단어 확률**을 출력하도록 하면, 단어를 하나씩 이어붙여 문장을 생성할 수 있습니다.

| 단계 | 모델 | 입력 → 출력 |
|------|------|-------------|
| 우리가 만든 것 | LSTM 분류 | 문장 전체 → 긍정/부정 (1개 값) |
| 한 단계 발전 | LSTM 생성 | 지금까지의 단어들 → 다음 단어 확률 분포 |
| 최종 형태 | GPT (Transformer) | 프롬프트 → 단어를 하나씩 이어서 문장 완성 |

### 5-2. 핵심 정리

```
딥러닝이 텍스트의 순서와 문맥을 이해할 수 있게 됨 (RNN/LSTM)
    ↓
Transformer(Self-Attention)로 더 빠르고 정확하게 발전
    ↓
대규모 텍스트로 '다음 단어 예측' 학습 → GPT
    ↓
사용자 질문에 단어를 하나씩 이어 생성 → ChatGPT
```

결국 ChatGPT는 오늘 우리가 배운 딥러닝 구조(임베딩 → 순서 처리 → 분류/생성)가
극도로 고도화된 결과물입니다.


---
## 6. 전체 핵심 정리

### 오늘 배운 개념 정리표

| 개념 | 핵심 | PyTorch 코드 |
|------|------|-------------|
| 신경망 | 입력→가중합→활성화→출력 | `nn.Linear` + `nn.ReLU` |
| 손실 함수 | 예측과 정답의 차이 | `nn.BCELoss()` |
| 역전파 | 기울기를 계산하여 가중치 업데이트 | `loss.backward()` |
| 임베딩 | 단어 정수 → 밀집 벡터 | `nn.Embedding` |
| RNN | 순서를 반영하며 문맥 누적 | `nn.RNN` |
| LSTM | RNN + 장기기억 (게이트 메커니즘) | `nn.LSTM` |

### 학습 루프 핵심 5단계

```python
for epoch in range(EPOCHS):
    predictions = model(X)           # 1. 순전파
    loss = criterion(predictions, y) # 2. 손실 계산
    optimizer.zero_grad()            # 3. 기울기 초기화
    loss.backward()                  # 4. 역전파
    optimizer.step()                 # 5. 가중치 업데이트
```

### ChatGPT까지의 발전 경로

```
BoW/TF-IDF (단어 빈도) → Word2Vec (의미 벡터) → RNN/LSTM (순서 이해)
  → Transformer (병렬 처리 + Self-Attention) → GPT (대규모 학습) → ChatGPT
```
